# Prompting Pipeline: Extracting Variables from NSW Coronial Findings (final)

This notebook sends coronial findings reports (PDFs) to an OpenAI model and asks it to fill in 28 variables for each report (for example age, cause of death, medications and several yes/no flags). It tries three **prompting techniques**:

| Technique | What the model is given |
|---|---|
| **Zero-shot** | Instructions and the report. No examples. |
| **Few-shot** | The same instructions, plus one or two *worked examples* (a report together with its correct answer), then the report to extract. |
| **Chain-of-thought (CoT)** | The same instructions, plus a request to reason step by step before giving the answer. |

Each API call is called a **trial**: one report, one prompt, one repeat. Every trial is saved to an **audit log** that records the exact prompt text, the model's answer and the number of tokens used. The audit log is the main output of this notebook; it is read by the evaluation notebook (`evaluation_pipeline_final.ipynb`).

## How the notebook is organised

| Section | What it holds |
|---|---|
| 1 to 4 | Setup, the 28-variable schema, the reports, and the pieces that prompts are built from. |
| 5 | How a prompt reaches the model (system, user and assistant messages). |
| 6 | What is saved for each trial, and how **changes to a prompt are recorded automatically** (prompt versions). |
| 7 | Three small helper functions shared by all techniques: the dataset check, the API call and the logging. |
| 8 | **Zero-shot:** its messages, its trial function, its prompts, the smoke test and the run. |
| 9 | **Few-shot:** the same, plus the worked examples. |
| 10 | **Chain-of-thought:** the same. |
| 11 to 12 | Export of the results, and the steps after prompt development. |

## The three sets of reports

The 50 reports are split into three sets, and the notebook only lets the current set run (Section 3).

| Set | Reports | Repeats per prompt and report | What it is used for |
|---|---|---|---|
| **Prompt development** | 5 | 5 | Every prompt is tried and compared here. |
| **Internal validation** | 5 | 5 | The best prompt from each of the three techniques (for example one zero-shot, one few-shot and one chain-of-thought prompt) is run here, and the single best of those three is chosen. |
| **External validation** | 40 | 1 | Kept **unseen** until the end. Only the single best prompt from internal validation is run on it, once. |

## Test mode

`TEST_MODE = True` runs only on one test report that is not part of the 50 study reports, so the pipeline can be checked without touching the study reports. Test results are saved in a separate `results_test/` folder. For the real experiment, set `MODEL` to the chosen full model and `TEST_MODE = False`.

## Requirements

Python 3.10 or later with `openai`, `python-dotenv`, `pydantic`, `pandas` and `openpyxl`; an OpenAI API key in a `.env` file (`OPENAI_API_KEY`); and the workbook `report_dataset_final_v8.xlsx` in the same folder (it holds the report links in the sheet `report_urls`).

## 1. Setup

Imports, the API key from the `.env` file, and the two settings that matter most: `MODEL` and `TEST_MODE`. The request settings are fixed for the whole study: `REASONING_EFFORT = "medium"` (the documented default of the chosen model) and `MAX_OUTPUT_TOKENS = 30000`. `TEMPERATURE`, `TOP_P` and `VERBOSITY` are left unset, so the model's defaults apply (temperature and top_p cannot be combined with a reasoning effort other than "none"). Whatever is set applies to every trial so that prompts are the only thing that differs, and it is saved with each trial. Section 1.1 checks which settings the chosen model accepts.

In [2]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
from pathlib import Path
from datetime import datetime
import hashlib
import json
import os
import time
import pandas as pd

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment (for example in a .env file) before running this notebook.")

# ---- THE MODEL ----------------------------------------------------------------------------------
# One model is used for the whole study. Candidates (confirm the exact name on the OpenAI model page):
#   "gpt-5.6-luna", "gpt-5.6-terra", "gpt-5.4", "gpt-5.5"
# Testing the pipeline: a cheap model and TEST_MODE = True. Real experiment: the chosen model and TEST_MODE = False.
# MODEL = "gpt-5.4-mini-2026-03-17"
MODEL = "gpt-5.6-luna"
TEST_MODE = True

# ---- REQUEST SETTINGS (identical for every trial) -------------------------------------------------
# The project protocol lists settings that are held constant across all prompts, so that the prompt is the only thing
# that differs. None means "not sent": the model's own default applies. Whatever is set here is saved with every trial
# (see PromptTrialLog), together with what the API reports it applied.
REASONING_EFFORT = "medium"    # fixed for the whole study: the documented default of gpt-5.6-luna (none, low, medium, high, xhigh and max exist)
TEMPERATURE = None             # protocol: 0.0 to 0.1. Not sent: OpenAI states that temperature and top_p have to be removed
TOP_P = None                   #   when the reasoning effort is not "none" (check_parameters() in Section 1.1 shows what the model accepts)
MAX_OUTPUT_TOKENS = 30000      # hard cap, identical for every trial; hidden reasoning tokens count towards it (test runs used under 3,000)
VERBOSITY = None               # "low" / "medium" / "high"; None = the model's default
# The Responses API has no `seed` parameter and its response has no `system_fingerprint` field (both belong to the Chat Completions API). The dated model name that served each call is saved instead (`model` in PromptTrialLog).

if not TEST_MODE and "mini" in MODEL:
    raise RuntimeError("TEST_MODE is False but MODEL is a mini model: set MODEL to the chosen full model for the real experiment.")

if not TEST_MODE and REASONING_EFFORT is None:
    raise RuntimeError("TEST_MODE is False but REASONING_EFFORT is None: set one fixed level in Section 1 so that it is identical for every trial (protocol, Section 3.4.3).")

WORKBOOK = "report_dataset_final_v8.xlsx"                      # holds the report links (sheet `report_urls`)
RESULTS_ROOT = Path("results_test") if TEST_MODE else Path("results")

client = OpenAI(api_key=OPENAI_API_KEY, timeout=600, max_retries=3)   # the SDK retries temporary errors by itself
print(f"MODEL = {MODEL} | TEST_MODE = {TEST_MODE} | results are saved in: {RESULTS_ROOT}/")

Loading .env from: /Users/prernaroy/Library/CloudStorage/OneDrive-UNSW/Year 2/disso/dissertation/.env
MODEL = gpt-5.6-luna | TEST_MODE = True | results are saved in: results_test/


### 1.1 Parameter check (text only, run once for each model)

`check_parameters()` sends a few very short, text-only requests (no reports and no study data) to the chosen model, each with one request setting added, and prints which settings the API accepts or rejects. The check is needed because the protocol's sampling settings (`temperature`, `top_p`) are rejected by many reasoning models, and the Responses API has no `seed` parameter. The result is also appended to `parameter_check.csv` in the results folder, so that the settings actually available to the model are documented. About seven very short calls are made.

In [3]:
def check_parameters() -> pd.DataFrame:
    """Test which request settings the chosen model accepts, using very short text-only requests.

    One request is sent without any setting (baseline) and one for each setting
    listed below. A setting that the API rejects is reported with the error
    message. Nothing is saved with the trials; the result is printed and
    appended to `parameter_check.csv` in the results folder.

    Returns:
        One row per tested setting: whether it was accepted, and what the API
        reported (or the error message).
    """
    candidates = {
        "no setting (baseline)": {},
        "reasoning effort = low": {"reasoning": {"effort": "low"}},
        "temperature = 0": {"temperature": 0},
        "top_p = 1": {"top_p": 1},
        "verbosity = low": {"text": {"verbosity": "low"}},
        "max_output_tokens = 2000": {"max_output_tokens": 2000},
        "seed = 1": {"extra_body": {"seed": 1}},
    }
    rows = []
    for name, kwargs in candidates.items():
        try:
            r = client.responses.create(model=MODEL, input="Reply with the single word: ready.", **kwargs)
            reasoning = getattr(r, "reasoning", None)
            detail = (f"model={r.model}; status={getattr(r, 'status', None)}; temperature={getattr(r, 'temperature', None)}; "
                      f"top_p={getattr(r, 'top_p', None)}; reasoning effort={getattr(reasoning, 'effort', None)}")
            rows.append({"setting": name, "accepted": True, "detail": detail})
        except Exception as e:
            message = str(getattr(e, "message", e)).replace("\n", " ")
            rows.append({"setting": name, "accepted": False, "detail": message[:200]})
    result = pd.DataFrame(rows)
    print(result.to_string(index=False))
    saved = result.assign(model=MODEL, checked_at=str(datetime.now()))
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    path = RESULTS_ROOT / "parameter_check.csv"
    saved.to_csv(path, mode="a", header=not path.exists(), index=False)
    return result

In [4]:
check_parameters();

                 setting  accepted                                                                                                                                                                                    detail
   no setting (baseline)      True                                                                                                model=gpt-5.6-luna; status=completed; temperature=1.0; top_p=0.98; reasoning effort=medium
  reasoning effort = low      True                                                                                                   model=gpt-5.6-luna; status=completed; temperature=1.0; top_p=0.98; reasoning effort=low
         temperature = 0     False Error code: 400 - {'error': {'message': "Unsupported parameter: 'temperature' is not supported with this model.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': None}}
               top_p = 1     False             Error code: 400 - {'error': {'message': "Unsupported parameter: 'top_

## 2. Schema: the 28 variables

`VariablesToExtractExtended` lists the 28 variables. It is passed to the API as the required output format (`text_format`, see Section 5), so every answer comes back as valid, correctly typed JSON whichever prompt produced it.

- Yes/no flags are `bool | None`: **True** or **False** only when the report says so explicitly, and `None` (blank) when it does not address the topic.
- Variables with several values (medications, conditions, health issues) are lists of text; the rest are text in the report's own wording.

`ExtractionWithReasoning` is a second schema used only for chain-of-thought (Section 10). It adds a `reasoning` field next to the same 28-variable extraction.

In [5]:
class VariablesToExtractExtended(BaseModel):
    file_number: str                       # single string (reports with >1 file number are excluded)
    sex: str
    age_at_death: str                      # str: can be an estimate/range, e.g. "47 or 48"
    date_of_findings: str                  # str: report's own wording
    date_of_death: str                     # str: report's own wording (e.g. "April 2013", a range, or blank)
    place_of_death: str
    definitive_cause_of_death: str
    potential_causes_of_death: list[str]
    manner_of_death: str                   # coroner's wording without name and pronouns (subjectless); blank if none
    country_of_birth: str
    place_of_residence: str                # most recent residence as stated; a region is acceptable
    profession: str                        # most recent, if mentioned, else blank
    level_of_education: str
    medications: list[str]
    intellectual_disability_flag: bool | None
    intellectual_disability_conditions: list[str]   # renamed from _labels; verbatim
    mental_health_flag: bool | None
    mental_health_conditions: list[str]             # renamed from _labels; verbatim
    substance_use_flag: bool | None        # abuse, misuse or dependence on alcohol/drugs, or smoking (see definitions_final)
    substances_used: list[str]
    known_health_issues: list[str]
    custody_served_flag: bool | None
    non_english_speaker_flag: bool | None
    use_of_interpreters_flag: bool | None
    preventable_death_flag: bool | None
    domestic_violence_flag: bool | None    # stays in the study (discuss its reliability in the report)
    toxicology_results_summary: str        # verbatim (name and pronouns removed), blank if none
    coroners_recommendations_summary: str  # verbatim incl. recipient heading (name and pronouns removed), blank if none


class ExtractionWithReasoning(BaseModel):
    """Chain-of-thought output shape: the model's reasoning, followed by the same 28-variable extraction."""
    reasoning: str
    extraction: VariablesToExtractExtended

## 3. Reports and which set is allowed to run

The report links come from the sheet `report_urls` of the workbook: one row per report with its set (`prompt_development`, `internal_validation` or `external_validation`), its file number and its link. The file number lets every trial check that the link opened the intended report.

**`ALLOWED_DATASETS` is the only switch.** While it contains just `"prompt_development"`, a trial on any other report stops with an error before anything is sent to the API. The switch is moved on deliberately, in this order:

1. Prompt development only (the setting below).
2. After the best prompt of each technique has been chosen, add `"internal_validation"` and run those three prompts.
3. After the single best prompt has been chosen, add `"external_validation"` and run that one prompt, once per report. The external reports are not used before this point.

Because every trial stores the full text of its prompt, the prompt run on the external set can be checked against the one chosen on the internal set.

In [6]:
def load_reports() -> dict[int, dict]:
    """Read the report list from the `report_urls` sheet of the workbook.

    Returns:
        A dict mapping each report number to a dict with the keys `dataset`,
        `file_number` and `file_url` (None when the sheet gives no link).
    """
    df = pd.read_excel(WORKBOOK, sheet_name="report_urls", dtype=str).fillna("")
    return {int(r["Report No."]): dict(dataset=r["Dataset"].strip(), file_number=r["file_number"].strip(),
                                       file_url=r["file_url"].strip() or None)
            for _, r in df.iterrows()}

REPORTS = load_reports()

# One extra report used only for testing the pipeline. It is not one of the 50 study reports (file 2017/00335331).
TEST_REPORT_NO = 900
REPORTS[TEST_REPORT_NO] = dict(dataset="test_report", file_number="2017/00335331",
    file_url="https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf")

**Which sets may run, and how many repeats each gets.** `ALLOWED_DATASETS` is moved on deliberately (prompt development, then internal validation, then external validation). `REPEATS_PER_DATASET` gives the number of repeats of each prompt on each report: 5 for prompt development and internal validation, 1 for external validation. `DATASET` is the set that is run; changing it and running the cell again updates `TARGET_REPORTS` and `N_REPEATS` as well.

In [7]:
# ---- Which sets may run -------------------------------------------------------------------------
ALLOWED_DATASETS = {"test_report"} if TEST_MODE else {"test_report", "prompt_development"}
# Later: ALLOWED_DATASETS.add("internal_validation")   # after the best prompt of each technique is chosen
# Later: ALLOWED_DATASETS.add("external_validation")   # after the single best prompt is chosen

# ---- Repeats of every prompt on every report, for each set --------------------------------------
REPEATS_PER_DATASET = {"test_report": 5, "prompt_development": 5, "internal_validation": 5, "external_validation": 1}

# ---- The set that is run now (change DATASET, then run this cell again) -------------------------
DATASET = "test_report" if TEST_MODE else "prompt_development"


def reports_in(dataset: str) -> list[int]:
    """Return the report numbers that belong to one set.

    Args:
        dataset: Name of the set, for example "prompt_development".

    Returns:
        The report numbers of that set, in ascending order.
    """
    return sorted(n for n, r in REPORTS.items() if r["dataset"] == dataset)


TARGET_REPORTS = reports_in(DATASET)
N_REPEATS = REPEATS_PER_DATASET[DATASET]
print(f"{DATASET}: reports {TARGET_REPORTS}, {N_REPEATS} repeat(s) of each prompt on each report")
for _d in ["prompt_development", "internal_validation", "external_validation"]:
    _missing = [n for n in reports_in(_d) if not REPORTS[n]["file_url"]]
    print(f"  {_d}: {len(reports_in(_d))} reports" + (f", links missing for {_missing}" if _missing else ", all links present"))

test_report: reports [900], 5 repeat(s) of each prompt on each report
  prompt_development: 5 reports, all links present
  internal_validation: 5 reports, all links present
  external_validation: 40 reports, all links present


## 4. Prompt building blocks

A prompt is built from the blocks below. Each entry of `COMPONENTS` is a separate piece of instruction text that can be switched on or off, which is what allows prompts to be compared. `TASK_CORE` is the short task statement (what to extract and what to do with missing information). `COT_INSTRUCTION` is added only for chain-of-thought.

The text of `variable_definitions` is the same wording as the sheet `definitions_final` in the workbook, which was also used to annotate the ground truth.

In [8]:
# Definitions: must be identical in wording to the sheet `definitions_final` of `report_dataset_final_v8.xlsx`.
# The same wording is used for the manual annotation of the ground truth and in the prompt; update the sheet and this dictionary together.
DECEDENT_RULE = ("For manner_of_death and toxicology_results_summary, use the coroner's own wording but remove the deceased's name and any title, pronouns or other references to the deceased, so that the answer reads as a subjectless phrase. This rule does not apply to coroners_recommendations_summary, which is copied exactly as written, including any name of the deceased. All other fields are recorded as stated in the report.")

BOOLEAN_RULE = ("For every boolean (*_flag) field below: return TRUE only if the report explicitly and directly confirms the fact, and FALSE only if the report explicitly and directly denies or negates it. If the report does not address the topic at all, leave the field blank. Do not return FALSE by default, and do not infer a value from indirect or contextual information, or from sources outside of the provided report.")

EMPTY_RULE = ("If the report does not provide the information for a field, leave the field empty: return null for a boolean flag, an empty string for a text field, and an empty list for a list field. Do not write placeholders such as 'N/A', 'unknown' or 'not stated'.")

FIELD_DEFINITIONS = {
    "file_number": "The unique case identifier assigned by the coroner's court. The format is typically a year followed by a case number, with a forward slash separator.",
    "sex": "The deceased's sex, only if explicitly stated or consistently indicated through pronoun use. Do not infer from a name.",
    "age_at_death": "The deceased's age in whole years at time of death, from an explicit statement or calculable from an explicitly stated date of birth and date of death. If only an estimated age or age range is given, record the estimate exactly as stated in the report.",
    "date_of_findings": "The date the coroner delivered or published the findings, as given by the date of findings field in the report. Record it as text in the report's own format and wording (for example '3 March 2020'); do not reformat it.",
    "date_of_death": "The date, estimated date, or date range the deceased died, as stated by the coroner. Preserve any qualifying language (e.g. \"on or about\") exactly rather than resolving it to a single date. Record it as text in the report's own format and wording; do not reformat it.",
    "place_of_death": "The location where the deceased died, or was found if the exact location of death is unknown. Record the location as specifically as the report states it, in the report's own wording.",
    "definitive_cause_of_death": "The specific, medically or forensically ascertained cause of death, stated by the coroner as an established fact. Leave blank if the coroner does not reach a definite conclusion. Use potential_causes_of_death instead if no definitive cause of death is given and potential causes are given in the report directly. Never populate both fields for the same report.",
    "potential_causes_of_death": "A list of possible, unconfirmed causes of death stated by the coroner, used only when no definitive cause is reached. The potential causes should be directly referenced within the report; do not make up or infer potential causes. Do not populate definitive_cause_of_death if no definitive cause is given and only potential causes are stated in the report.",
    "manner_of_death": "The coroner's stated manner of death, in the coroner's own wording, with the decedent's name and pronouns removed so that it reads as a subjectless phrase (see the general rule on decedent references). If no manner of death is given in the report, do not infer one from the rest of the report; leave it empty.",
    "country_of_birth": "The country the deceased was born in, if explicitly stated. Do not infer from surname, cultural references, or place of residence.",
    "place_of_residence": "The deceased's most recent place of residence before death, exactly as stated in the report. A region or area is acceptable if the report gives nothing more specific.",
    "profession": "The deceased's most recent occupation or status (e.g. a stated profession, or \"retired\"/\"unemployed\"/\"student\" if that was their status at time of death).",
    "level_of_education": "The deceased's highest confirmed educational achievement. If a qualification was started but not completed, state this explicitly rather than recording it as achieved.",
    "medications": "A list of every medication that the report describes the deceased as taking or being prescribed, at any time. Do not include a substance only because it was detected in toxicology screening unless the report also describes it as a prescribed or taken medication.",
    "intellectual_disability_flag": "Whether the report explicitly documents a diagnosed or clearly described intellectual, developmental, or learning disability.",
    "intellectual_disability_conditions": "If intellectual_disability_flag is true, the specific condition(s) named, using the report's own terminology.",
    "mental_health_flag": "Whether the report explicitly documents a diagnosed or clearly described mental health condition.",
    "mental_health_conditions": "If mental_health_flag is true, the specific condition(s) named, using the report's own terminology.",
    "substance_use_flag": "Whether the report explicitly states that the deceased abused, misused or was dependent on alcohol or any other drug (for example a diagnosed substance use disorder, addiction, a drug or alcohol problem, or heavy, regular or problematic use), or smoked tobacco or used nicotine products. Social, occasional or experimental use of alcohol or other drugs does not count. A medication taken as prescribed does not count, and a substance mentioned only in toxicology results, without a statement about the deceased's use of it, does not count on its own. False only if the report explicitly states that the deceased did not misuse such substances.",
    "substances_used": "If substance_use_flag is true, the alcohol, tobacco and other drugs that the report names as abused, misused or relied on by the deceased (and tobacco or nicotine where the deceased smoked or used it), using the report's own terminology. If the substance is not named, give the general term the report uses followed by '(unspecified)'. Leave empty otherwise.",
    "known_health_issues": "A list of the deceased's known health conditions, including physical or chronic conditions, injuries and health-related circumstances, and including any conditions listed under intellectual_disability_conditions or mental_health_conditions. Each item should be a short phrase. Conditions that result from substance use (for example alcohol withdrawal) are health issues and are included. Do not include substance use or dependence itself, or lifestyle behaviours such as smoking or drinking.",
    "custody_served_flag": "Whether the report explicitly states the deceased served a custodial sentence or was held in custody at any point, including police or correctional custody at the time of death. Being charged, arrested, or having outstanding warrants alone does not qualify.",
    "non_english_speaker_flag": "Whether the report explicitly states the deceased was not a native or fluent English speaker, or had English as an additional language. Do not infer this from country of birth or name alone.",
    "use_of_interpreters_flag": "Whether the report explicitly states an interpreter was used at any point connected to the case.",
    "preventable_death_flag": "Whether the coroner explicitly and directly states the death was preventable. The coroner making recommendations alone does not satisfy this. An explicit statement on preventability is required.",
    "domestic_violence_flag": "Whether the report describes a domestic violence related charge or conviction involving the deceased, an apprehended domestic violence order involving the deceased, or a direct statement that the deceased experienced or perpetrated domestic violence.",
    "toxicology_results_summary": "The toxicology findings copied from the report in the coroner's own wording, including the substances detected and the levels at which they were found, with the deceased's name, title and any pronouns removed (see the general rule on decedent references). If no toxicology findings are included in the report, leave empty.",
    "coroners_recommendations_summary": "The coroner's formal recommendations copied verbatim from the report, covering every distinct recommendation and including any name of the deceased exactly as written. Include the heading naming the recipient(s) of the recommendations (for example 'To the Chief Executive, ...') and keep the original numbering. Do not include general observations or notes that are not formal recommendations. If the report makes no recommendations, leave empty.",
}

# Safety check: every schema field must have a definition (and vice versa), so code and workbook cannot drift apart silently.
assert list(FIELD_DEFINITIONS) == list(VariablesToExtractExtended.model_fields), \
    "FIELD_DEFINITIONS and the schema are out of sync"

def _build_variable_definitions() -> str:
    """Join the general rules and the field definitions into one block of text.

    Returns:
        DECEDENT_RULE, BOOLEAN_RULE and EMPTY_RULE, followed by one
        "field: definition" entry per field of FIELD_DEFINITIONS, separated by spaces.
    """
    parts = [DECEDENT_RULE, BOOLEAN_RULE, EMPTY_RULE]
    parts += [f"{name}: {text}" for name, text in FIELD_DEFINITIONS.items()]
    return " ".join(parts)

COMPONENTS = {
    "role_definition": (
        "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."
    ),

    "mhc_id_icd_definitions": (
        "According to the ICD-11, a mental health condition is a clinically significant disturbance in cognition, emotional regulation, or behaviour that reflects dysfunction in psychological, biological, or developmental processes and causes distress or impairment. An intellectual disability is defined as a developmental condition characterised by significant limitations in intellectual functioning (reasoning, problem-solving, learning) and adaptive behaviour, originating during the developmental period."
    ),

    "date_format_instruction": (
        "Report all dates exactly as they appear in the source document; do not reformat or infer missing date components."
    ),

    "acronym_instruction": (
        "For diagnostic labels, use the full clinical term rather than an acronym (e.g., 'Attention-Deficit/Hyperactivity Disorder', not 'ADHD'), unless the source document uses only the acronym and gives no full form."
    ),

    "variable_definitions": _build_variable_definitions(),

    # (the old '2-5 sentence summary' wording was removed because toxicology/recommendations are verbatim)
    "detail_level_instruction": (
        "For list-type fields (conditions, medications, known health issues, potential causes of death), keep each item to a short phrase found in the report rather than a full sentence. For all other fields, extract only what is explicitly stated, without inferred elaboration."
    ),
}

In [9]:
# The field list is generated from the schema so the task text can never drift from it.
_FIELD_LIST = ", ".join(VariablesToExtractExtended.model_fields)

TASK_CORE = (
    f"Extract the following medical and demographic information from the given coronial findings and recommendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string (or an empty list for list fields), and null for boolean flags. Outputs must be accurate and concise; do not provide additional commentary or explanation. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Extract the following fields: {_FIELD_LIST}."
)


COT_INSTRUCTION = (
    "Before producing your final answer, reason step by step through the report to determine each field's value. Put that reasoning in the `reasoning` field, then put the completed extraction in the `extraction` field."
)

## 5. How a prompt reaches the model

Every trial is one call of `client.responses.parse()`, set up as in the OpenAI guide on structured outputs ([developers.openai.com/api/docs/guides/structured-outputs](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=responses)):

- `input` is a list of **messages**, each with a `role` and some `content`.
- `text_format` is the Pydantic schema from Section 2, which forces the answer into the 28 variables.

The roles used here, with the names used throughout this project:

| Role | Contains | Earlier name in this project |
|---|---|---|
| **system message** (`"role": "system"`) | The standing instructions: the role, the formatting rules and the definitions (the `COMPONENTS`). | "top slot" |
| **user message** (`"role": "user"`) | The task text (`TASK_CORE`) and the report PDF. | "user slot" / "user content" |
| **assistant message** (`"role": "assistant"`) | Used in few-shot only: the correct answer to an example report. | |

OpenAI's text-generation guide calls the instruction role `developer`; the structured-outputs guide uses `system`. The API accepts both, and the `instructions=` parameter does the same job. This project follows the structured-outputs guide.

**The report PDF.** A PDF is added to a user message as `{"type": "input_file", "file_url": "<link>"}`. The API downloads the PDF from the link and passes the model **both the extracted text and an image of the pages** ([PDF guide](https://developers.openai.com/api/docs/guides/pdf-files)). This is why one report can be tens of thousands of tokens. A potential workaround is to use existing Python libraries such as `pypdf` to extract the text layer of the PDFs to decrease the tokens used.

### What each technique sends

The three layouts below are built by `build_zero_shot_messages()`, `build_few_shot_messages()` and `build_cot_messages()` (Sections 8 to 10), one per technique, so that each layout can be read on its own.

**ZERO-SHOT**
```
system:    instructions
user:      task text + REPORT (via file_url)
```

**FEW-SHOT**
```
system:    instructions
user:      task text + EXAMPLE 1 (via file_url)
assistant: EXAMPLE 1's correct answer
user:      EXAMPLE 2 (via file_url)
assistant: EXAMPLE 2's correct answer
user:      REPORT (via file_url)
```

**CHAIN-OF-THOUGHT**
```
system:    instructions
user:      task text + reasoning request + REPORT (via file_url)
```

Note that zero-shot prompt Z0 is the exception: it has no system message and no task text, so the user message holds only the report PDF.

`compose()` below turns the switched-on components into the system message text and the user message text. `show_messages()` prints the layout of any message list (the role of each message and each part), without calling the API. Long texts are shortened to 90 characters; `show_messages(messages, max_chars=None)` shows them in full.

In [10]:
def compose(flags: dict, include_task_core: bool = True) -> tuple[str, str]:
    """Build the system and user text of a prompt from the switched-on components.

    The components that are True in `flags` are joined into the system message;
    the user message holds the task text. The report PDF is added later, by the
    build_*_messages functions.

    Args:
        flags: Component name -> True/False, selecting entries of COMPONENTS.
        include_task_core: Whether the user text holds TASK_CORE (otherwise it is empty).

    Returns:
        A (system_text, user_text) tuple.
    """
    system_text = " ".join(text for name, text in COMPONENTS.items() if flags.get(name))
    user_text = TASK_CORE if include_task_core else ""
    return system_text, user_text


def show_messages(messages: list[dict], max_chars: int | None = 90) -> None:
    """Print the layout of a message list without sending anything to the API.

    For each message, the role is printed with each text part (and its word
    count) or the link of each PDF part. Long texts are shortened unless
    `max_chars` is None.

    Args:
        messages: The message list, as built by a build_*_messages function.
        max_chars: How many characters of each text to show. Use None to show
            the whole text, for example the full system message.
    """
    def short(t):
        """Return `t` (quoted, shortened to `max_chars` if set) followed by its word count."""
        shown = t if max_chars is None or len(t) <= max_chars else t[:max_chars] + "..."
        return repr(shown) + f"  [{len(t.split())} words]"
    for m in messages:
        role = f"{m['role']:>9}:"
        if isinstance(m["content"], str):
            print(role, short(m["content"]))
            continue
        for part in m["content"]:
            what = f"input_text  {short(part['text'])}" if part["type"] == "input_text" else f"input_file  {part['file_url']}"
            print(role, what)
            role = " " * 10

## 6. What is saved for every trial, and how prompt changes are recorded

`PromptTrialLog` is the record saved for each trial: the exact prompt text, the report, the model's answer, and the number of tokens used. Token counts are recorded so that prompts can be compared on how much they use (a longer prompt, extra examples or reasoning all increase them). `usage_cached_tokens` is the part of the input that OpenAI served from its prompt cache. `reasoning` is filled only for chain-of-thought trials.

Each record is written to its own folder (`trials/<trial_id>/trial.json`) and appended to one file for the set (`audit_log.jsonl`), which is the file the evaluation notebook reads.

**Editing prompts during prompt development.** The wording of a prompt can be edited at any time (the components in Section 4, the task text, the chain-of-thought request, the worked examples, or the choice of components in Sections 8 to 10). Every trial stores the exact text it was sent, and in addition the notebook keeps track of the *versions* of each prompt automatically:

- A prompt name (for example `Z4`) starts at **version 1**.
- If the text sent under that name later differs in any way (system message, task text, chain-of-thought request, worked examples), the next trial is recorded as **version 2**, then 3, and so on. The full text of each version is stored in `prompt_versions.jsonl`, together with the date and a free-text note (`RUN_NOTE`, set before each run).
- Returning to an earlier wording reuses that earlier version number.
- Each trial stores `prompt_version` and `prompt_hash` (a short code of the exact text). The evaluation notebook scores `Z4_v1` and `Z4_v2` as different prompts, so results obtained with different wordings are never mixed.

No prompt needs to be renamed after an edit, and nothing has to be recorded by hand.

**Request settings.** The settings that the protocol holds constant across prompts are saved with every trial: `model` (the dated model name returned by the API), `reasoning_effort`, `temperature`, `top_p`, `max_output_tokens` and `verbosity` (as set in Section 1; `None` means the setting was not sent and the model's default applied). Alongside these are `api_reported_settings` (what the API reports it applied), `service_tier`, `response_status` and `incomplete_reason` (filled if an answer was cut short). The Responses API has neither a `seed` parameter nor a `system_fingerprint` field, so neither can be recorded.

In [11]:
class PromptTrialLog(BaseModel):
    """The record saved for one trial: what was sent, what came back, and how many tokens it used."""
    trial_id: str
    dataset: str                            # prompt_development / internal_validation / external_validation / test_report
    report_no: int
    expected_file_number: str               # from the workbook
    file_number_match: bool | None = None   # did the model read the file number we expected? (checks the link)
    technique: str                          # zero_shot / few_shot / chain_of_thought
    n_shots: int = 0                        # number of worked examples (few-shot only)
    prompt_id: str                          # the name of the prompt, e.g. Z4, FS2
    prompt_version: int = 0                 # 1, 2, ... = the wording of this prompt name (0 = test report, not versioned)
    prompt_hash: str = ""                   # short code of the exact prompt text; same code = same text
    repeat_index: int
    test_mode: bool = False                 # True = run with the test setup; never part of the experiment
    factors: dict                           # which components were switched on
    include_task_core: bool
    system_text: str                        # the exact text of the system message ("" = none)
    user_text: str                          # the exact text of the user message, before the PDF ("" = none)
    example_urls: list[str] = []            # the PDFs used as worked examples (few-shot only)
    file_url: str                           # the report PDF
    notes: str = ""
    response_id: str
    created_at: str
    completed_at: str
    latency_seconds: float
    model: str | None
    reasoning_effort: str | None = None     # request settings, as set in Section 1; None = not sent (the model's default)
    temperature: float | None = None
    top_p: float | None = None
    max_output_tokens: int | None = None
    verbosity: str | None = None
    api_reported_settings: dict = {}        # what the API reports it applied: temperature, top_p, max_output_tokens, reasoning_effort, verbosity
    service_tier: str | None = None         # the service tier that processed the request, as returned by the API
    response_status: str | None = None      # completed / incomplete / failed
    incomplete_reason: str | None = None    # why an answer was cut short (for example max_output_tokens)
    usage_input_tokens: int                 # includes the PDF(s)
    usage_cached_tokens: int = 0            # part of the input served from the prompt cache
    usage_output_tokens: int                # includes any hidden reasoning tokens
    usage_reasoning_tokens: int = 0
    usage_total_tokens: int
    output: dict                            # the 28 extracted variables
    reasoning: str | None = None            # chain-of-thought text (chain-of-thought only)

The cell below keeps the list of prompt versions in `prompt_versions.jsonl` (in the results folder). `resolve_prompt_version()` is called once for each trial: it compares the exact prompt text with the versions already recorded for that prompt name, and either returns the matching version or records a new one. A prompt text that is new in internal or external validation produces a warning, because only the prompts tried in prompt development are meant to be run there. `show_prompt_versions()` lists all versions, and `show_prompt_version("Z4", 1)` prints the full text of one version (to restore an earlier wording).

In [12]:
PROMPT_VERSIONS_FILE = RESULTS_ROOT / "prompt_versions.jsonl"


def prompt_fingerprint(technique: str, system_text: str, user_text: str, examples: list[dict]) -> str:
    """Return a short code that identifies the exact prompt.

    The same technique, text and worked examples always give the same code, so
    the code shows whether a prompt has changed between runs.

    Args:
        technique: The prompting technique, for example "zero_shot".
        system_text: The system message.
        user_text: The user message (before the PDF).
        examples: The worked examples (file_url and ideal_output of each).

    Returns:
        The first 10 hexadecimal characters of the SHA-256 hash of the prompt.
    """
    content = {"technique": technique, "system": system_text, "user": user_text,
               "examples": [{"file_url": e["file_url"], "ideal_output": e["ideal_output"]} for e in examples]}
    return hashlib.sha256(json.dumps(content, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:10]


def read_prompt_versions() -> list[dict]:
    """Read every recorded prompt version from PROMPT_VERSIONS_FILE.

    Returns:
        One dict per recorded version, in the order recorded; an empty list
        if the file does not exist yet.
    """
    if not PROMPT_VERSIONS_FILE.exists():
        return []
    return [json.loads(line) for line in open(PROMPT_VERSIONS_FILE) if line.strip()]


def resolve_prompt_version(prompt_id: str, technique: str, system_text: str, user_text: str,
                           examples: list[dict], dataset: str, note: str = "") -> tuple[int, str]:
    """Return the version of the prompt that was just sent, recording it if it is new.

    A prompt text seen before keeps its version. A new text gets the next version
    number and is appended to PROMPT_VERSIONS_FILE, except on the test report,
    where no version is created (version 0).

    Args:
        prompt_id: The prompt name, for example "Z4".
        technique: The prompting technique.
        system_text: The system message.
        user_text: The user message (before the PDF).
        examples: The worked examples (empty for zero-shot and chain-of-thought).
        dataset: The set the trial belongs to.
        note: Free-text note stored with a newly recorded version.

    Returns:
        A (version, prompt_hash) tuple.
    """
    prompt_hash = prompt_fingerprint(technique, system_text, user_text, examples)
    known = [v for v in read_prompt_versions() if v["prompt_id"] == prompt_id]
    for v in known:
        if v["prompt_hash"] == prompt_hash:
            return v["version"], prompt_hash                          # this exact text has been used before
    if dataset == "test_report":
        return 0, prompt_hash                                        # test runs never create versions
    version = max((v["version"] for v in known), default=0) + 1
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    with open(PROMPT_VERSIONS_FILE, "a") as f:
        f.write(json.dumps(dict(prompt_id=prompt_id, version=version, prompt_hash=prompt_hash, technique=technique,
                                n_shots=len(examples), first_used=str(datetime.now()), first_dataset=dataset, note=note,
                                system_text=system_text, user_text=user_text, examples=examples), ensure_ascii=False) + "\n")
    if known:
        print(f"  NOTE: the text of prompt {prompt_id} differs from its earlier version(s): recorded as {prompt_id}_v{version}.")
    if dataset in ("internal_validation", "external_validation"):
        print(f"  WARNING: {prompt_id}_v{version} is a new wording that was not run in prompt development. "
              f"Use the wording that was chosen (see show_prompt_versions()).")
    return version, prompt_hash


def show_prompt_versions() -> pd.DataFrame:
    """Return a table of the recorded prompt versions, one row per version.

    Returns:
        A DataFrame with the prompt name, version, technique, number of examples,
        word count, hash, first set and first use of each version; empty if
        nothing has been recorded.
    """
    versions = pd.DataFrame(read_prompt_versions())
    if versions.empty:
        return versions
    versions["words"] = versions["system_text"].str.split().str.len() + versions["user_text"].str.split().str.len()
    return versions[["prompt_id", "version", "technique", "n_shots", "words", "prompt_hash", "first_dataset", "first_used", "note"]]


def show_prompt_version(prompt_id: str, version: int) -> None:
    """Print the full text of one recorded prompt version (to restore an earlier wording).

    Args:
        prompt_id: The prompt name, for example "Z4".
        version: The version number.

    Raises:
        StopIteration: If that version has not been recorded.
    """
    v = next(v for v in read_prompt_versions() if v["prompt_id"] == prompt_id and v["version"] == version)
    print(f"{prompt_id}_v{version} ({v['technique']}, {v['n_shots']} example(s), first used {v['first_used']}, note: {v['note']!r})")
    print("\n--- system message ---\n" + (v["system_text"] or "(none)"))
    print("\n--- user message (before the PDF) ---\n" + (v["user_text"] or "(none)"))
    for i, e in enumerate(v["examples"], 1):
        print(f"\n--- worked example {i}: {e['file_url']} ---\n" + json.dumps(e["ideal_output"], ensure_ascii=False)[:300] + " ...")

In [13]:
def dataset_dir(dataset: str) -> Path:
    """Return the results folder of one set, creating it if needed.

    Args:
        dataset: Name of the set.

    Returns:
        The path RESULTS_ROOT / dataset.
    """
    d = RESULTS_ROOT / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_trial(dataset: str, log_entry: PromptTrialLog) -> None:
    """Save one trial to its own folder and append it to the set's audit log.

    Writes `trials/<trial_id>/trial.json` and adds one line to `audit_log.jsonl`
    in the set's results folder.

    Args:
        dataset: Name of the set.
        log_entry: The trial record to save.
    """
    t_dir = dataset_dir(dataset) / "trials" / log_entry.trial_id
    t_dir.mkdir(parents=True, exist_ok=True)
    (t_dir / "trial.json").write_text(log_entry.model_dump_json(indent=2))
    with open(dataset_dir(dataset) / "audit_log.jsonl", "a") as f:
        f.write(log_entry.model_dump_json() + "\n")

## 7. Helper functions shared by the three techniques

Each technique has its own trial function (Sections 8 to 10) that reads in the same order: **check the set, build the messages, call the model, log the trial.** Three of those steps are the same for every technique and are written once here:

- `check_trial_allowed()` checks that the report may be run now (the right set, an allowed set, a link present). Nothing is sent to the API if the check fails.
- `call_model()` makes the API call (`client.responses.parse()`) with the model and the output format.
- `log_trial()` turns the response into a `PromptTrialLog` (including the prompt version), saves it and returns it. `print_trial()` prints one line per trial.

The messages are the one step that differs between techniques, so they are built in separate functions in Sections 8 to 10.

In [14]:
def check_trial_allowed(report_no: int, dataset: str) -> dict:
    """Check, before anything is sent to the API, that this report may be run now.

    Args:
        report_no: The report number.
        dataset: The set the trial is meant to belong to.

    Returns:
        The report's entry in REPORTS.

    Raises:
        KeyError: If the report is not in the `report_urls` sheet.
        PermissionError: If the set is not in ALLOWED_DATASETS, the report belongs
            to another set, or the report has no link.
    """
    if report_no not in REPORTS:
        raise KeyError(f"Report {report_no} is not in the `report_urls` sheet.")
    report = REPORTS[report_no]
    if dataset not in ALLOWED_DATASETS:
        raise PermissionError(f"The set '{dataset}' is not allowed to run yet (ALLOWED_DATASETS = {sorted(ALLOWED_DATASETS)}).")
    if report["dataset"] != dataset:
        raise PermissionError(f"Report {report_no} belongs to '{report['dataset']}', not '{dataset}'.")
    if not report["file_url"]:
        raise PermissionError(f"Report {report_no} has no link in the `report_urls` sheet.")
    return report

In [15]:
def settings_in_use() -> dict:
    """Return the request settings that are set in Section 1 (those that are not None).

    The names are the ones used in PromptTrialLog.

    Returns:
        A dict of setting name -> value.
    """
    chosen = {"reasoning_effort": REASONING_EFFORT, "temperature": TEMPERATURE, "top_p": TOP_P,
              "max_output_tokens": MAX_OUTPUT_TOKENS, "verbosity": VERBOSITY}
    return {name: value for name, value in chosen.items() if value is not None}


def api_reported_settings(response) -> dict:
    """Return the settings that the API reports it applied to a response.

    Args:
        response: The API response from call_model.

    Returns:
        A dict with temperature, top_p, max_output_tokens, reasoning_effort and
        verbosity (None where the response does not report a value).
    """
    reasoning = getattr(response, "reasoning", None)
    text_config = getattr(response, "text", None)
    return {"temperature": getattr(response, "temperature", None), "top_p": getattr(response, "top_p", None),
            "max_output_tokens": getattr(response, "max_output_tokens", None),
            "reasoning_effort": getattr(reasoning, "effort", None), "verbosity": getattr(text_config, "verbosity", None)}


def call_model(messages: list[dict], text_format):
    """Make one API call and time it, with the request settings from Section 1.

    Args:
        messages: The message list to send.
        text_format: The schema the answer must follow.

    Returns:
        A (response, seconds) tuple.
    """
    settings = settings_in_use()
    extra = {name: settings[name] for name in ("temperature", "top_p", "max_output_tokens") if name in settings}
    if "reasoning_effort" in settings:
        extra["reasoning"] = {"effort": settings["reasoning_effort"]}
    if "verbosity" in settings:
        extra["text"] = {"verbosity": settings["verbosity"]}
    start = time.time()
    response = client.responses.parse(model=MODEL, input=messages, text_format=text_format, **extra)
    return response, time.time() - start


def log_trial(response, seconds: float, *, report_no: int, dataset: str, technique: str, prompt_id: str, repeat_index: int,
              flags: dict, include_task_core: bool, system_text: str, user_text: str,
              examples: list[dict] | None = None, use_cot: bool = False, notes: str = "") -> dict:
    """Turn an API response into a saved trial record.

    The response is parsed into a PromptTrialLog, the prompt version is resolved
    (and recorded if new), and the trial is saved. A warning is printed if the
    file number read by the model differs from the expected one.

    Args:
        response: The API response from call_model.
        seconds: The duration of the call.
        report_no: The report number.
        dataset: The set the trial belongs to.
        technique: The prompting technique.
        prompt_id: The prompt name.
        repeat_index: The repeat number of this prompt on this report.
        flags: The prompt components that were switched on.
        include_task_core: Whether the task text was included.
        system_text: The system message that was sent.
        user_text: The user message that was sent (before the PDF).
        examples: The worked examples, for few-shot prompts.
        use_cot: Whether the answer holds `reasoning` and `extraction` parts.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.
    """
    examples = examples or []
    report = REPORTS[report_no]
    parsed = json.loads(response.output_text)
    reasoning, output = (parsed["reasoning"], parsed["extraction"]) if use_cot else (None, parsed)
    usage = response.usage
    cached = getattr(getattr(usage, "input_tokens_details", None), "cached_tokens", 0) or 0
    reasoning_tokens = getattr(getattr(usage, "output_tokens_details", None), "reasoning_tokens", 0) or 0
    version, prompt_hash = resolve_prompt_version(prompt_id, technique, system_text, user_text, examples, dataset, notes)

    log_entry = PromptTrialLog(
        trial_id=f"{datetime.now().strftime('%Y%m%dT%H%M%S%f')}_{response.id[-8:]}",
        dataset=dataset, report_no=report_no, expected_file_number=report["file_number"],
        file_number_match=(str(output.get("file_number", "")).strip() == report["file_number"]),
        technique=technique, n_shots=len(examples), prompt_id=prompt_id, prompt_version=version, prompt_hash=prompt_hash,
        repeat_index=repeat_index, test_mode=TEST_MODE, factors=flags, include_task_core=include_task_core,
        system_text=system_text, user_text=user_text, example_urls=[e["file_url"] for e in examples],
        file_url=report["file_url"], notes=notes, response_id=response.id,
        created_at=str(datetime.fromtimestamp(response.created_at)),
        completed_at=str(datetime.fromtimestamp(response.completed_at)) if getattr(response, "completed_at", None) else "",
        latency_seconds=round(seconds, 4), model=response.model, **settings_in_use(),
        usage_input_tokens=usage.input_tokens, usage_cached_tokens=cached, usage_output_tokens=usage.output_tokens,
        usage_reasoning_tokens=reasoning_tokens, usage_total_tokens=usage.total_tokens,
        output=output, reasoning=reasoning, api_reported_settings=api_reported_settings(response),
        service_tier=getattr(response, "service_tier", None), response_status=getattr(response, "status", None),
        incomplete_reason=getattr(getattr(response, "incomplete_details", None), "reason", None),
    )
    save_trial(dataset, log_entry)
    if not log_entry.file_number_match:
        print(f"  WARNING: report {report_no}: the model read file number {output.get('file_number')!r}, expected {report['file_number']!r}. Check the link.")
    if log_entry.response_status not in (None, "completed"):
        print(f"  WARNING: report {report_no}: the response status is {log_entry.response_status!r} ({log_entry.incomplete_reason}); the answer may be cut short.")
    return log_entry.model_dump()


def print_trial(record: dict) -> None:
    """Print a one-line summary of a saved trial (prompt, repeat, tokens and trial id).

    Args:
        record: The trial record returned by a run_*_trial function.
    """
    label = f"{record['prompt_id']}_v{record['prompt_version']}" if record["prompt_version"] else record["prompt_id"]
    print(f"report {record['report_no']} | {label} repeat {record['repeat_index']}: {record['usage_input_tokens']} in "
          f"({record['usage_cached_tokens']} cached) / {record['usage_output_tokens']} out tokens -> {record['trial_id']}")

## 8. Zero-shot

### 8.1 The messages

Zero-shot sends a system message (the instructions, left out when a prompt has none) and one user message holding the task text (also left out when a prompt has none) and the report PDF. The report is added as an `input_file` with its `file_url`.

In [16]:
def build_zero_shot_messages(system_text: str, user_text: str, file_url: str) -> list[dict]:
    """Build the message list of a zero-shot prompt.

    Layout:
        system:  the instructions (left out when empty)
        user:    the task text (left out when empty) + the REPORT, as a PDF link

    Args:
        system_text: The system message.
        user_text: The task text.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    user_content = []
    if user_text:
        user_content.append({"type": "input_text", "text": user_text})
    user_content.append({"type": "input_file", "file_url": file_url})
    messages.append({"role": "user", "content": user_content})
    return messages

### 8.2 The zero-shot trial function

Four steps: check the set, build the messages, call the model, log the trial.

In [17]:
def run_zero_shot_trial(report_no: int, prompt_id: str, flags: dict, include_task_core: bool = True, *,
                        dataset: str, repeat_index: int = 1, notes: str = "") -> dict:
    """Run one zero-shot call on one report and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        include_task_core: Whether to include the task text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.
    """
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    messages = build_zero_shot_messages(system_text, user_text, report["file_url"])          # 2. messages
    response, seconds = call_model(messages, VariablesToExtractExtended)                     # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="zero_shot",   # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, notes=notes)

### 8.3 The zero-shot prompts (edit here)

The prompts add one kind of content at a time, so the effect of each can be seen. The system message holds the components; the user message holds the task text and the report.

| Prompt | System message | User message |
|---|---|---|
| **Z0** | none | the report only (no instructions at all: the model sees the schema and the PDF) |
| **Z1** | none | task text + report |
| **Z2** | role, date format, acronyms, level of detail | task text + report |
| **Z3** | Z2 + definitions of a mental health condition and an intellectual disability | task text + report |
| **Z4** | Z3 + the definition of every variable and the rules for yes/no flags | task text + report |

Z1 is not an empty prompt: the task text tells the model what to extract, to leave missing information empty and to give no commentary.

**To try a different wording**, edit the text in Section 4 or change the components of a prompt below, and run again. The edit is recorded automatically as a new version of that prompt (Section 6). A new prompt can be added as a new line, for example `Z5`.

In [18]:
Z2_FLAGS = {"role_definition": True, "date_format_instruction": True, "acronym_instruction": True, "detail_level_instruction": True}
Z3_FLAGS = {**Z2_FLAGS, "mhc_id_icd_definitions": True}
Z4_FLAGS = {**Z3_FLAGS, "variable_definitions": True}

# prompt_id -> which components are switched on, and whether the task text is included
ZERO_SHOT_PROMPTS = {
    "Z0": {"flags": {},       "include_task_core": False},
    "Z1": {"flags": {},       "include_task_core": True},
    "Z2": {"flags": Z2_FLAGS, "include_task_core": True},
    "Z3": {"flags": Z3_FLAGS, "include_task_core": True},
    "Z4": {"flags": Z4_FLAGS, "include_task_core": True},
}
ZERO_SHOT_TO_RUN = list(ZERO_SHOT_PROMPTS)
if TEST_MODE:        # the test report only needs one prompt per technique to test the pipeline and the agreement metrics
    ZERO_SHOT_TO_RUN = ["Z4"]
print("zero-shot prompts to run:", ZERO_SHOT_TO_RUN)            # or a shorter list, e.g. ["Z4"]

zero-shot prompts to run: ['Z4']


**Preview.** The layout of the messages for Z4, without calling the API:

In [19]:
_p = ZERO_SHOT_PROMPTS["Z4"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_zero_shot_messages(_system, _user, REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


   system: 'You are an expert at structured data extraction. You will be given unstructured text from ...'  [1448 words]
     user: input_text  'Extract the following medical and demographic information from the given coronial findings...'  [113 words]
           input_file  https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf


### 8.4 Smoke test: one call before anything else

One real call (prompt Z4) on the test report. Run it in test mode (`TEST_MODE = True`) after changing `MODEL`, to confirm that the model name works, that a PDF link is accepted and how many tokens a call uses. It is saved in the `test_report` folder and is not part of any study set.

The cell runs in test mode only; with `TEST_MODE = False` it prints a note and does nothing, so that *Run All* can be used for the real run.

In [20]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped for the real run
    _p = ZERO_SHOT_PROMPTS["Z4"]
    record = run_zero_shot_trial(TEST_REPORT_NO, "Z4", _p["flags"], _p["include_task_core"], dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| file number matched:", record["file_number_match"])
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out, {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

model: gpt-5.6-luna | file number matched: True
tokens: 46903 in (0 cached), 1920 out, 48823 total | 18.5276 s


### 8.5 Run the zero-shot prompts

Settings come from Section 3: `DATASET`, `TARGET_REPORTS` and `N_REPEATS` (5 repeats in prompt development and internal validation, 1 in external validation). Every report is run with every prompt in `ZERO_SHOT_TO_RUN` (in test mode only Z4, set in Section 8.3; run that cell again after changing `TEST_MODE`), and each combination is repeated `N_REPEATS` times. `RUN_NOTE` is an optional free-text note that is saved with every trial and with any new prompt version (for example what was changed and why).

In [21]:
RUN_NOTE = ""        # optional, e.g. "Z4: added a rule about smoking"

for report_no in TARGET_REPORTS:
    for prompt_id in ZERO_SHOT_TO_RUN:
        prompt = ZERO_SHOT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_zero_shot_trial(report_no, prompt_id, prompt["flags"], prompt["include_task_core"],
                                         dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE)
            print_trial(record)

report 900 | Z4 repeat 1: 46903 in (46900 cached) / 2286 out tokens -> 20261023T235721769882_c49df8e2
report 900 | Z4 repeat 2: 46903 in (46900 cached) / 2491 out tokens -> 20261023T235742528133_85575cec
report 900 | Z4 repeat 3: 46903 in (46900 cached) / 2297 out tokens -> 20261023T235803188058_77a192a3
report 900 | Z4 repeat 4: 46903 in (46900 cached) / 2193 out tokens -> 20261023T235822697639_dd83d46a
report 900 | Z4 repeat 5: 46903 in (46900 cached) / 2282 out tokens -> 20261023T235842813386_dd33aea5


## 9. Few-shot

### 9.1 How it works

**How it works with several `file_url`s.** A `file_url` is only a web link to a PDF. In every few-shot call the API downloads each linked PDF and gives the model its text and page images, exactly as for the report being extracted. A 2-shot call therefore contains **three PDFs**, in this order:

1. user message: the task text and **example report 1**
2. assistant message: example 1's correct answer (the dictionary below)
3. user message: **example report 2**
4. assistant message: example 2's correct answer
5. user message: **the report to extract**, which the model answers

The model is not shown a "file of examples": it is shown the example reports and their correct answers as an earlier part of the conversation, then asked to do the same for the last report. The examples are sent again in every call, so a 1-shot call uses roughly twice and a 2-shot call roughly three times the input tokens of a zero-shot call (the combined size of the PDFs in one request must stay under 50 MB, which is not a concern for these reports).

**The examples.** `FEW_SHOT_EXAMPLES` holds the two example reports (rows EX1 and EX2 of the sheet `few_shot_examples`), each as the link to its PDF and the correct answer in the same form as the model's output (the 28 variables). They are written out here so that they can be read and edited directly. Neither example is one of the 50 study reports.

In [22]:
FEW_SHOT_EXAMPLES = [
    {
        # EX1: file 2021/196736 (annotated by hand from this report; not one of the 50 study reports)
        "file_url": "https://coroners.nsw.gov.au/documents/findings/2025/Inquest_into_the_Death_of_Frank_Dwayne_COLEMAN.pdf",
        "ideal_output": {
            "file_number": "2021/196736",
            "sex": "Male",
            "age_at_death": "43",
            "date_of_findings": "22 October 2025",
            "date_of_death": "between 8:00pm on 7 July 2021 and 5:24am on 8 July 2021",
            "place_of_death": "Metropolitan Special Programs Centre at Long Bay Correctional Complex",
            "definitive_cause_of_death": "methadone toxicity",
            "potential_causes_of_death": [],
            "manner_of_death": "Died after consuming methadone that was supplied by another inmate",
            "country_of_birth": "Australia",
            "place_of_residence": "",
            "profession": "",
            "level_of_education": "Ceased university shortly after joining",
            "medications": ["Aripiprazole", "Atorvastatin", "Bisoprolol", "Entresto (sacubitril / valsartan)", "Escitalopram", "Europagliflozin", "Metformin", "Rabeprazole", "Spironolactone", "Zyprexa"],
            "intellectual_disability_flag": None,
            "intellectual_disability_conditions": [],
            "mental_health_flag": True,
            "mental_health_conditions": ["chronic schizophrenia", "paranoid schizophrenia"],
            "substance_use_flag": True,
            "substances_used": ["alcohol", "tobacco", "cannabis", "heroin", "methamphetamine"],
            "known_health_issues": ["chronic schizophrenia", "paranoid schizophrenia", "cardiomyopathy", "congestive cardiac failure", "gastro-oesophageal reflux disease (GORD)", "hepatitis B", "hepatitis C", "hepatitis D", "hypercholesterolaemia", "hypertension", "thiamine deficiency"],
            "custody_served_flag": True,
            "non_english_speaker_flag": None,
            "use_of_interpreters_flag": None,
            "preventable_death_flag": None,
            "domestic_violence_flag": True,
            "toxicology_results_summary": "Toxicological analysis detected methadone in the blood at 0.41 mg/L, a level within the reported lethal range. Aripiprazole was detected at a supratherapeutic level, along with therapeutic type concentrations of citalopram, metformin and olanzapine. Low levels of atorvastatin and bisoprolol were also detected.",
            "coroners_recommendations_summary": "To the Chief Executive, Justice Health and Forensic Mental Health Network NSW (Justice Health)/ the Commissioner, Corrective Services NSW\n1. Justice Health, in coordination with Corrective Services NSW, review the models of care and resourcing of Drug and Alcohol assessments for the commencement of opioid agonist therapy (OAT), with a view to increasing access and capacity, and\na. Produce a joint report estimating unmet need in the public jail system and outlining the optimum wait time\nb. Produce a joint mapping report indicating what resources would be necessary to achieve the optimum wait time (including staffing and infrastructure needs)\n\nJustice Health, in coordination with Corrective Services NSW:\na. assess the viability of Drug and Alcohol reviews being conducted via telehealth, including in-cell, and if deemed viable\nb. trial providing Drug and Alcohol reviews by in-cell telehealth",
        },
    },
    {
        # EX2: file 2023/74097 (annotated by hand from this report; not one of the 50 study reports)
        "file_url": "https://coroners.nsw.gov.au/documents/findings/2024/FURNER_Brett_-_Findings.pdf",
        "ideal_output": {
            "file_number": "2023/74097",
            "sex": "Male",
            "age_at_death": "54",
            "date_of_findings": "25 June 2024",
            "date_of_death": "5 March 2023",
            "place_of_death": "Goulburn Base Hospital, Goulburn NSW 2580",
            "definitive_cause_of_death": "acute myocardial infarction due to coronary artery atherosclerotic disease",
            "potential_causes_of_death": [],
            "manner_of_death": "Died of natural causes whilst in the lawful custody of the New South Wales Police Force",
            "country_of_birth": "",
            "place_of_residence": "",
            "profession": "Church Minister",
            "level_of_education": "theological college",
            "medications": [],
            "intellectual_disability_flag": None,
            "intellectual_disability_conditions": [],
            "mental_health_flag": True,
            "mental_health_conditions": ["depression", "bipolar disorder", "delusional disorder"],
            "substance_use_flag": True,
            "substances_used": ["alcohol", "tobacco"],
            "known_health_issues": ["depression", "bipolar disorder", "delusional disorder", "easily angered", "delusional behaviour", "paranoid behaviour", "struck by a vehicle", "triple vessel coronary artery disease and severe left ventricular dysfunction", "hypertension", "alcohol withdrawal", "critical stenosis of the left anterior descending coronary artery", "elevated blood pressure", "abnormal liver function", "myocardial infarction", "back injury"],
            "custody_served_flag": True,
            "non_english_speaker_flag": None,
            "use_of_interpreters_flag": None,
            "preventable_death_flag": None,
            "domestic_violence_flag": None,
            "toxicology_results_summary": "toxicological analysis identified significant intoxication at the time of death (blood alcohol level of 0.171 g/100mL), with low levels of diazepam and its active metabolite",
            "coroners_recommendations_summary": "",
        },
    },
]

In [23]:
# Checks: each answer must fit the schema, and no example may be one of the 50 study reports.
_study_file_numbers = {r["file_number"] for n, r in REPORTS.items() if n != TEST_REPORT_NO}
for _ex in FEW_SHOT_EXAMPLES:
    VariablesToExtractExtended(**_ex["ideal_output"])                      # raises an error if a variable is missing or mistyped
    assert _ex["ideal_output"]["file_number"] not in _study_file_numbers, "an example must not be a study report"
print(f"{len(FEW_SHOT_EXAMPLES)} examples OK:", [e["ideal_output"]["file_number"] for e in FEW_SHOT_EXAMPLES])

2 examples OK: ['2021/196736', '2023/74097']


### 9.2 The messages

The system message holds the instructions. The task text appears once, in the first user message, next to the first worked example. Each example is a user message (the example PDF) followed by an assistant message (its correct answer, written as JSON). The report to extract is always the last user message.

In [24]:
def build_few_shot_messages(system_text: str, user_text: str, examples: list[dict], file_url: str) -> list[dict]:
    """Build the message list of a few-shot prompt.

    Layout:
        system:     the instructions (left out when empty)
        user:       the task text + EXAMPLE 1, as a PDF link
        assistant:  EXAMPLE 1's correct answer
        user:       EXAMPLE 2, as a PDF link
        assistant:  EXAMPLE 2's correct answer
        user:       the REPORT to extract, as a PDF link

    Args:
        system_text: The system message.
        user_text: The task text (placed once, before the first example).
        examples: The worked examples, each with `file_url` and `ideal_output`.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    for i, example in enumerate(examples):
        user_content = []
        if i == 0 and user_text:
            user_content.append({"type": "input_text", "text": user_text})                 # the task text appears once, first
        user_content.append({"type": "input_file", "file_url": example["file_url"]})
        messages.append({"role": "user", "content": user_content})
        messages.append({"role": "assistant", "content": json.dumps(example["ideal_output"])})

    messages.append({"role": "user", "content": [{"type": "input_file", "file_url": file_url}]})
    return messages

### 9.3 The few-shot trial function

In [25]:
def run_few_shot_trial(report_no: int, prompt_id: str, flags: dict, examples: list[dict], include_task_core: bool = True, *,
                       dataset: str, repeat_index: int = 1, notes: str = "") -> dict:
    """Run one few-shot call on one report, with the given worked examples, and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        examples: The worked examples (at least one).
        include_task_core: Whether to include the task text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.

    Raises:
        AssertionError: If no worked example is given.
    """
    assert examples, "few-shot needs at least one worked example"
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    messages = build_few_shot_messages(system_text, user_text, examples, report["file_url"]) # 2. messages
    response, seconds = call_model(messages, VariablesToExtractExtended)                     # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="few_shot",     # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, examples=examples, notes=notes)

### 9.4 The few-shot prompts (edit here)

FS1 gives one worked example and FS2 gives two. Each prompt has its own instructions (`flags`), which are set to the full zero-shot prompt (Z4) here but can be any other set of components. A change to the instructions, to the task text or to a worked example is recorded automatically as a new version of that prompt (Section 6).

In [26]:
# prompt_id -> the instructions (components), whether the task text is included, and the number of worked examples
FEW_SHOT_PROMPTS = {
    "FS1": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 1},
    "FS2": {"flags": Z4_FLAGS, "include_task_core": True, "n_shots": 2},
}
FEW_SHOT_TO_RUN = list(FEW_SHOT_PROMPTS)
if TEST_MODE:        # the test report only needs one prompt per technique to test the pipeline and the agreement metrics
    FEW_SHOT_TO_RUN = ["FS2"]
print("few-shot prompts to run:", FEW_SHOT_TO_RUN)

few-shot prompts to run: ['FS2']


**Preview.** The layout of the messages for FS2, without calling the API:

In [27]:
_p = FEW_SHOT_PROMPTS["FS2"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_few_shot_messages(_system, _user, FEW_SHOT_EXAMPLES[:_p["n_shots"]], REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


   system: 'You are an expert at structured data extraction. You will be given unstructured text from ...'  [1448 words]
     user: input_text  'Extract the following medical and demographic information from the given coronial findings...'  [113 words]
           input_file  https://coroners.nsw.gov.au/documents/findings/2025/Inquest_into_the_Death_of_Frank_Dwayne_COLEMAN.pdf
assistant: '{"file_number": "2021/196736", "sex": "Male", "age_at_death": "43", "date_of_findings": "2...'  [302 words]
     user: input_file  https://coroners.nsw.gov.au/documents/findings/2024/FURNER_Brett_-_Findings.pdf
assistant: '{"file_number": "2023/74097", "sex": "Male", "age_at_death": "54", "date_of_findings": "25...'  [165 words]
     user: input_file  https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf


### 9.5 Smoke test: one few-shot call

One real call (prompt FS2, with both worked examples) on the test report. A few-shot call sends three PDF links (two worked examples and the report), so this confirms that such a request is accepted and shows how many tokens it uses. Run it once after the zero-shot smoke test (Section 8.4), and again after changing `MODEL`. It is saved in the `test_report` folder and is not part of any study set.

The cell runs in test mode only; with `TEST_MODE = False` it prints a note and does nothing, so that *Run All* can be used for the real run.

In [28]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped for the real run
    _p = FEW_SHOT_PROMPTS["FS2"]
    record = run_few_shot_trial(TEST_REPORT_NO, "FS2", _p["flags"], FEW_SHOT_EXAMPLES[:_p["n_shots"]], _p["include_task_core"],
                                dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| worked examples:", _p["n_shots"], "| file number matched:", record["file_number_match"])
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out, {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

model: gpt-5.6-luna | worked examples: 2 | file number matched: True
tokens: 82333 in (2510 cached), 2393 out, 84726 total | 22.646 s


### 9.6 Run the few-shot prompts

In test mode only FS2 runs (`FEW_SHOT_TO_RUN`, set in Section 9.4; run that cell again after changing `TEST_MODE`). The number of repeats comes from Section 3 (5 for the test report).

In [29]:
RUN_NOTE = ""        # optional, e.g. "FS2: instructions changed to Z3"

for report_no in TARGET_REPORTS:
    for prompt_id in FEW_SHOT_TO_RUN:
        prompt = FEW_SHOT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_few_shot_trial(report_no, prompt_id, prompt["flags"], FEW_SHOT_EXAMPLES[:prompt["n_shots"]],
                                        prompt["include_task_core"], dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE)
            print_trial(record)

report 900 | FS2 repeat 1: 82333 in (82330 cached) / 2025 out tokens -> 20261023T235922396463_1a9ba083
report 900 | FS2 repeat 2: 82333 in (82330 cached) / 2074 out tokens -> 20261023T235941545270_d1fd1c92
report 900 | FS2 repeat 3: 82333 in (82330 cached) / 1993 out tokens -> 20261024T000001002776_0a9473d8
report 900 | FS2 repeat 4: 82333 in (82330 cached) / 2257 out tokens -> 20261024T000022299197_cd8f20a4
report 900 | FS2 repeat 5: 82333 in (82330 cached) / 2217 out tokens -> 20261024T000043341068_f93dbd91


## 10. Chain-of-thought

### 10.1 The messages

The layout is the same as zero-shot, but the user message also holds the request to reason step by step (`COT_INSTRUCTION`, or another wording given to a prompt). The model first writes its reasoning in the `reasoning` field and then fills the 28 variables in `extraction` (the `ExtractionWithReasoning` schema, Section 2). The reasoning is extra output text, so chain-of-thought trials normally use more output tokens.

In [30]:
def build_cot_messages(system_text: str, user_text: str, file_url: str) -> list[dict]:
    """Build the message list of a chain-of-thought prompt.

    Layout:
        system:  the instructions (left out when empty)
        user:    the task text + the request to reason step by step (already
                 part of `user_text`) + the REPORT, as a PDF link

    Args:
        system_text: The system message.
        user_text: The task text, including the chain-of-thought instruction.
        file_url: The link to the report PDF.

    Returns:
        The message list.
    """
    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})

    user_content = []
    if user_text:
        user_content.append({"type": "input_text", "text": user_text})
    user_content.append({"type": "input_file", "file_url": file_url})
    messages.append({"role": "user", "content": user_content})
    return messages

### 10.2 The chain-of-thought trial function

The reasoning is stored in `reasoning` and the 28 variables in `output`, so `output` has the same shape for every technique.

In [31]:
def run_cot_trial(report_no: int, prompt_id: str, flags: dict, include_task_core: bool = True, cot_instruction: str = COT_INSTRUCTION, *,
                  dataset: str, repeat_index: int = 1, notes: str = "") -> dict:
    """Run one chain-of-thought call on one report and save it.

    Args:
        report_no: The report number.
        prompt_id: The prompt name.
        flags: The prompt components to switch on.
        include_task_core: Whether to include the task text.
        cot_instruction: The request to reason step by step, added to the user text.
        dataset: The set the trial belongs to.
        repeat_index: The repeat number of this prompt on this report.
        notes: Free-text note stored with the trial.

    Returns:
        The saved record, as a dict.
    """
    report = check_trial_allowed(report_no, dataset)                                         # 1. check
    system_text, user_text = compose(flags, include_task_core)
    user_text = (user_text + "\n\n" + cot_instruction).strip()                               # the request to reason step by step
    messages = build_cot_messages(system_text, user_text, report["file_url"])                # 2. messages
    response, seconds = call_model(messages, ExtractionWithReasoning)                        # 3. call
    return log_trial(response, seconds, report_no=report_no, dataset=dataset, technique="chain_of_thought",   # 4. log
                     prompt_id=prompt_id, repeat_index=repeat_index, flags=flags, include_task_core=include_task_core,
                     system_text=system_text, user_text=user_text, use_cot=True, notes=notes)

### 10.3 The chain-of-thought prompts (edit here)

Each prompt has its own instructions (`flags`) and its own request to reason (`cot_instruction`), so that the wording of the request can be varied. A change to either is recorded automatically as a new version of that prompt (Section 6).

In [32]:
# prompt_id -> the instructions (components), whether the task text is included, and the request to reason step by step
COT_PROMPTS = {
    "COT1": {"flags": Z4_FLAGS, "include_task_core": True, "cot_instruction": COT_INSTRUCTION},
}
COT_TO_RUN = list(COT_PROMPTS)
if TEST_MODE:        # the test report only needs one prompt per technique to test the pipeline and the agreement metrics
    COT_TO_RUN = ["COT1"]
print("chain-of-thought prompts to run:", COT_TO_RUN)

chain-of-thought prompts to run: ['COT1']


**Preview.** The layout of the messages for COT1, without calling the API:

In [33]:
_p = COT_PROMPTS["COT1"]
_system, _user = compose(_p["flags"], _p["include_task_core"])
show_messages(build_cot_messages(_system, (_user + "\n\n" + _p["cot_instruction"]).strip(), REPORTS[TEST_REPORT_NO]["file_url"]))
# To see the full texts, call show_messages(..., max_chars=None)


   system: 'You are an expert at structured data extraction. You will be given unstructured text from ...'  [1448 words]
     user: input_text  'Extract the following medical and demographic information from the given coronial findings...'  [146 words]
           input_file  https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf


### 10.4 Smoke test: one chain-of-thought call

One real call (prompt COT1) on the test report. Besides the model name, the PDF link and the token counts, this confirms that the reasoning is returned together with the 28 variables. Run it once after the zero-shot smoke test (Section 8.4), and again after changing `MODEL`. It is saved in the `test_report` folder and is not part of any study set.

The cell runs in test mode only; with `TEST_MODE = False` it prints a note and does nothing, so that *Run All* can be used for the real run.

In [34]:
if TEST_MODE:      # smoke tests run in test mode only (they use the test report); skipped for the real run
    _p = COT_PROMPTS["COT1"]
    record = run_cot_trial(TEST_REPORT_NO, "COT1", _p["flags"], _p["include_task_core"], _p["cot_instruction"],
                           dataset="test_report", notes="smoke test")
    print("model:", record["model"], "| file number matched:", record["file_number_match"], "| reasoning returned:", bool(record.get("reasoning")))
    print(f"tokens: {record['usage_input_tokens']} in ({record['usage_cached_tokens']} cached), {record['usage_output_tokens']} out ({record['usage_reasoning_tokens']} reasoning), {record['usage_total_tokens']} total | {record['latency_seconds']} s")
else:
    print("Smoke test skipped: TEST_MODE is False (the smoke tests belong to test mode).")

model: gpt-5.6-luna | file number matched: True | reasoning returned: True
tokens: 47021 in (0 cached), 2450 out (843 reasoning), 49471 total | 18.9883 s


### 10.5 Run the chain-of-thought prompts

In test mode only COT1 runs (`COT_TO_RUN`, set in Section 10.3; run that cell again after changing `TEST_MODE`). The number of repeats comes from Section 3 (5 for the test report).

In [35]:
RUN_NOTE = ""        # optional, e.g. "COT1: reasoning request reworded"

for report_no in TARGET_REPORTS:
    for prompt_id in COT_TO_RUN:
        prompt = COT_PROMPTS[prompt_id]
        for repeat_index in range(1, N_REPEATS + 1):
            record = run_cot_trial(report_no, prompt_id, prompt["flags"], prompt["include_task_core"], prompt["cot_instruction"],
                                   dataset=DATASET, repeat_index=repeat_index, notes=RUN_NOTE)
            print_trial(record)

report 900 | COT1 repeat 1: 47021 in (47018 cached) / 2169 out tokens -> 20261024T000118312896_f073951f
report 900 | COT1 repeat 2: 47021 in (47018 cached) / 2106 out tokens -> 20261024T000135620574_bf02708c
report 900 | COT1 repeat 3: 47021 in (47018 cached) / 2644 out tokens -> 20261024T000158069880_5ec55f7a
report 900 | COT1 repeat 4: 47021 in (47018 cached) / 2623 out tokens -> 20261024T000218936250_d29eae04
report 900 | COT1 repeat 5: 47021 in (47018 cached) / 2356 out tokens -> 20261024T000237444928_a1205e7a


## 11. Export: prompt tracker, review sheet, token usage and prompt versions

Reads every trial logged for `DATASET` and writes two spreadsheets to its results folder: `prompt_tracker.xlsx` (one row per trial: the prompt, its version and settings, and its token usage) and `human_review.xlsx` (one row per trial: the prompt version, the report, the full text of the prompt that was sent, the repeat, and one column per extracted variable, for reading the outputs). Run this cell again after more trials. The last lines summarise token usage per prompt version, and the list of all recorded prompt versions is shown below it.

In [36]:
log_df = pd.read_json(dataset_dir(DATASET) / "audit_log.jsonl", lines=True)
for _col, _default in [("prompt_version", 0), ("prompt_hash", ""), ("reasoning_effort", None), ("temperature", None), ("top_p", None),
                       ("max_output_tokens", None), ("verbosity", None), ("service_tier", None), ("response_status", None)]:                  # records from older pipeline versions
    if _col not in log_df:
        log_df[_col] = _default

factor_cols = pd.json_normalize(log_df["factors"]).add_prefix("factor__")
tracker_df = pd.concat(
    [log_df[["trial_id", "dataset", "report_no", "technique", "n_shots", "prompt_id", "prompt_version", "prompt_hash", "repeat_index", "model", "reasoning_effort", "temperature", "top_p", "max_output_tokens", "verbosity", "service_tier", "notes",
             "usage_input_tokens", "usage_cached_tokens", "usage_output_tokens", "usage_reasoning_tokens",
             "usage_total_tokens", "latency_seconds", "file_number_match", "response_status"]], factor_cols], axis=1)
tracker_df.to_excel(dataset_dir(DATASET) / "prompt_tracker.xlsx", index=False)


def full_prompt_text(row) -> str:
    """Return the full text of the prompt that was sent in one trial, as one string.

    The string holds the system message and the user message. For a few-shot trial,
    a final line states how many worked examples were sent (the examples are re-sent
    with every call) and the links of their reports. The text is cut at 32,000
    characters, the limit of one Excel cell.

    Args:
        row: One record of the audit log.

    Returns:
        The prompt text.
    """
    system = row["system_text"] if "system_text" in row and isinstance(row["system_text"], str) else ""
    user = row["user_text"] if isinstance(row["user_text"], str) else ""
    text = (f"SYSTEM MESSAGE:\n{system}\n\n" if system else "NO SYSTEM MESSAGE\n\n") + f"USER MESSAGE:\n{user}"
    if row["n_shots"] and row["n_shots"] > 0:
        urls = row["example_urls"] if isinstance(row["example_urls"], list) else []
        text += f"\n\n[{int(row['n_shots'])} worked example(s) sent before the report: " + "; ".join(map(str, urls)) + "]"
    return text[:32000]


output_df = pd.json_normalize(log_df["output"]).apply(lambda col: col.map(lambda v: "; ".join(map(str, v)) if isinstance(v, list) else v))
review_df = pd.concat(
    [log_df[["prompt_id", "prompt_version", "report_no"]],
     log_df.apply(full_prompt_text, axis=1).rename("prompt_text"),                                  # the full prompt, between report_no and repeat_index
     log_df[["repeat_index", "trial_id", "technique", "n_shots", "notes"]], output_df], axis=1
).sort_values(["prompt_id", "prompt_version", "report_no", "repeat_index", "trial_id"])
with pd.ExcelWriter(dataset_dir(DATASET) / "human_review.xlsx", engine="openpyxl") as _xw:
    review_df.to_excel(_xw, sheet_name="human_review", index=False)
    _ws = _xw.sheets["human_review"]
    _ws.freeze_panes = "A2"
    _ws.column_dimensions["D"].width = 60                                                           # the prompt text sits in column D

# average tokens per trial for each prompt version
_keys = ["technique", "prompt_id", "prompt_version"]
token_summary = (log_df.groupby(_keys)[["usage_input_tokens", "usage_cached_tokens", "usage_output_tokens", "usage_total_tokens"]]
                 .mean().round(0).astype(int).assign(trials=log_df.groupby(_keys).size()))
token_summary

,,,usage_input_tokens,usage_cached_tokens,usage_output_tokens,usage_total_tokens,trials
technique,prompt_id,prompt_version,,,,,
chain_of_thought,COT1,0,47021,43100,2381,49402,12
few_shot,FS2,0,82333,75678,2193,84526,12
zero_shot,Z4,0,46903,42992,2284,49188,12


In [37]:
show_prompt_versions()

""


## 12. After prompt development

1. Score the prompt-development results with `evaluation_pipeline_final.ipynb`. It ranks the prompt versions overall and within each technique (a prompt version is shown as, for example, `Z4_v2`) and reports the agreement between the repeats.
2. Choose the best prompt version of each technique (for example `Z4_v2`, `FS2_v1` and `COT1_v1`). The rule agreed in advance is the highest unweighted mean of the per-variable scores, with ties going to the shorter prompt.
3. Make sure the wording in the notebook is exactly the chosen wording. If the chosen version is not the latest, `show_prompt_version("Z4", 1)` prints its full text so that it can be restored.
4. Internal validation: add `"internal_validation"` to `ALLOWED_DATASETS`, set `DATASET = "internal_validation"` and run the cell in Section 3 again (this updates `TARGET_REPORTS` and sets `N_REPEATS` to 5). Set the `..._TO_RUN` lists so that only the three chosen prompts run, and run Sections 8 to 10 again. Each trial prints its prompt version; a warning appears if a wording is used that was not run in prompt development. Score the results and pick the single best of the three.
5. External validation: add `"external_validation"`, set `DATASET = "external_validation"` and run the cell in Section 3 again (`N_REPEATS` becomes 1), run only the chosen prompt, and score it.

Token usage per prompt version (Section 11) can be compared between the chosen prompts and the others once the results are in. OpenAI applies prompt caching automatically to repeated prompt beginnings (`usage_cached_tokens` shows how much of the input was served from the cache); it affects cost and speed and not the extracted values, and it is noted in the dissertation rather than analysed.